# Lecture 3 — Improving the Resume Scorer

In this notebook we score resumes against a job description and submit results to a shared leaderboard.

**Gold** resumes (prefixed `g`) are strong matches for the role. **Silver** resumes (prefixed `s`) are weaker. A good scoring strategy should clearly separate gold from silver.

In [7]:
from pydantic import BaseModel, Field
from resume_utils import load_resumes, load_job_requirements, analyze_resume, submit_score

OPENROUTER_API_KEY = "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"  # <-- Paste your OpenRouter API key here
TEAM_NAME = "my-team"    # <-- Change this to your team name

resumes = load_resumes('../data/resumes_final_with_gold_silver.csv')
job_req = load_job_requirements('../data/job_req_senior.md')

print(f"Loaded {len(resumes)} resumes")

Loaded 134 resumes


## Define the output schema and prompt

In [5]:
class ResumeScore(BaseModel):
    score: int = Field(..., ge=0, le=100, description="Resume match score from 0 to 100")
    reasoning: str = Field(..., description="Brief explanation of the score")

prompt = f"""Score this resume against the job requirements on a 0-100 scale.
Provide a score and brief reasoning.

Job Requirements:
{job_req}"""

## Score all resumes and submit to leaderboard

In [6]:
scores = {}
for i, (rid, resume) in enumerate(sorted(resumes.items())):
    print(f"  {i+1}/{len(resumes)}: {rid}", end=" ")
    resp = analyze_resume(
        api_key=OPENROUTER_API_KEY,
        prompt=prompt,
        resume_text=resume['Resume_str'],
        output_schema=ResumeScore,
    )
    if resp['result']:
        scores[rid] = resp['result']['score']
        print(f"-> {scores[rid]}")
    else:
        print(f"-> ERROR: {resp['error']}")
        scores[rid] = 0

    submit_score(TEAM_NAME, rid, scores[rid])

print(f"\nDone. Submitted {len(scores)} scores for team '{TEAM_NAME}'.")

  1/134: 10089434 -> ERROR: Client error '400 Bad Request' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400


HTTPStatusError: Client error '401 Unauthorized' for url 'http://ai-leaderboard.site/lecture3/api/submit'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

## Review results

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {'resume_id': rid, 'score': s,
     'tier': 'gold' if rid.startswith('g') else 'silver' if rid.startswith('s') else 'wild'}
    for rid, s in scores.items()
])

print(df.groupby('tier')['score'].agg(['mean', 'std', 'min', 'max']).round(1))
print(f"\nGold-Silver gap: {df[df.tier=='gold']['score'].mean() - df[df.tier=='silver']['score'].mean():.1f}")